In [1]:
from openai import OpenAI
import os 
from dotenv import load_dotenv
load_dotenv(override=True)

key=os.getenv('GROQ_API_KEY')
url = "https://api.groq.com/openai/v1"
myAi = OpenAI(api_key=key, base_url=url)
MODEL = "openai/gpt-oss-120b"

In [2]:
def messages_for(lang1_code,lang1,lang2):
    system_prompt = f"""
    Your task is to convert {lang1} code into high performance {lang2} code.
    Respond only with {lang2} code. Do not provide any explanation other than occasional comments.
    The {lang2} response needs to produce an identical output in the fastest possible time.
    """
    user_prompt = f"""
    Port this {lang1} code to {lang2} with the fastest possible implementation that produces identical output in the least time.
    Your response will be written to a file called Program
    Respond only with {lang2} code.
    {lang1} code to port:

    ```{lang1}
    {lang1_code}
    ```
    """
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [ ]:
def port(client, model, lang1_code, lang1, lang2):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(lang1_code, lang1, lang2),
        reasoning_effort=reasoning_effort
    )
    reply = response.choices[0].message.content or ""
    reply = reply.replace(f'```{lang2}', '').replace('```', '').strip()
    return reply


In [4]:
# Gradio UI - AI Code Converter
import gradio as gr

LANGUAGES = ["python", "java", "c++", "javascript", "typescript", "c", "c#", "go", "rust", "php", "kotlin", "swift"]

def convert_code(source_code, input_language, output_language):
    if not source_code or not source_code.strip():
        return "Please enter source code first."

    if input_language == output_language:
        return source_code

    try:
        return port(
            myAi,
            MODEL,
            source_code,
            input_language,
            output_language
        )
    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(title="AI Code Converter") as demo:

    gr.Markdown("# 🔄 AI Code Converter")
    gr.Markdown(
        "Convert code from one programming language to another using AI."
    )

    with gr.Row():

        # LEFT SIDE - INPUT
        with gr.Column():

            input_language = gr.Dropdown(
                choices=LANGUAGES,
                value="python",
                label="Input Language"
            )

            source_code = gr.Textbox(
                label="Source Code",
                placeholder="Paste your source code here...",
                lines=20,
                max_lines=30
            )

        # RIGHT SIDE - OUTPUT
        with gr.Column():

            output_language = gr.Dropdown(
                choices=LANGUAGES,
                label="Target Language"
            )

            target_code = gr.Textbox(
                label="Converted Code",
                placeholder="Converted code will appear here...",
                lines=20,
                max_lines=30
            )

    convert_button = gr.Button(
        "🚀 Convert Code",
        variant="primary"
    )

    convert_button.click(
        fn=convert_code,
        inputs=[
            source_code,
            input_language,
            output_language
        ],
        outputs=target_code
    )

    gr.Markdown(
        "**Tip:** Select the input and target languages, "
        "paste your code, then click Convert Code."
    )


demo.launch(share=True, inline=False)

C:\Users\jatin\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3d934e293e7c8e5438.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
